In [1]:
%%shell
# Ubuntu no longer distributes chromium-browser outside of snap
#
# Proposed solution: https://askubuntu.com/questions/1204571/how-to-install-chromium-without-snap

# Add debian buster
cat > /etc/apt/sources.list.d/debian.list <<'EOF'
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster.gpg] http://deb.debian.org/debian buster main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-buster-updates.gpg] http://deb.debian.org/debian buster-updates main
deb [arch=amd64 signed-by=/usr/share/keyrings/debian-security-buster.gpg] http://deb.debian.org/debian-security buster/updates main
EOF

# Add keys
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
apt-key adv --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A

apt-key export 77E11517 | gpg --dearmour -o /usr/share/keyrings/debian-buster.gpg
apt-key export 22F3D138 | gpg --dearmour -o /usr/share/keyrings/debian-buster-updates.gpg
apt-key export E562B32A | gpg --dearmour -o /usr/share/keyrings/debian-security-buster.gpg

# Prefer debian repo for chromium* packages only
# Note the double-blank lines between entries
cat > /etc/apt/preferences.d/chromium.pref << 'EOF'
Package: *
Pin: release a=eoan
Pin-Priority: 500


Package: *
Pin: origin "deb.debian.org"
Pin-Priority: 300


Package: chromium*
Pin: origin "deb.debian.org"
Pin-Priority: 700
EOF

# Install chromium and chromium-driver
apt-get update
apt-get install chromium chromium-driver

# Install selenium
pip install selenium

Executing: /tmp/apt-key-gpghome.dYWeCSWeWe/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys DCC9EFBF77E11517
gpg: key DCC9EFBF77E11517: public key "Debian Stable Release Key (10/buster) <debian-release@lists.debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.j5Inq1szyu/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 648ACFD622F3D138
gpg: key DC30D7C23CBBABEE: public key "Debian Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Executing: /tmp/apt-key-gpghome.WbTsEkNXZq/gpg.1.sh --keyserver keyserver.ubuntu.com --recv-keys 112695A0E562B32A
gpg: key 4DFAB270CAA96DFA: public key "Debian Security Archive Automatic Signing Key (10/buster) <ftpmaster@debian.org>" imported
gpg: Total number processed: 1
gpg:               imported: 1
Get:1 http://deb.debian.org/debian buster InRelease [122 kB]
Get:2 https://cloud.r-project.org/bin/l

In [2]:
#!pip install selenium==4.1.0
!pip install beautifulsoup4
!pip install lxml
!pip install pandas
!pip install webdriver-manager
!pip install python-dateutil
!pip install bottle
!pip install pivottablejs

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 3.5 MB/s eta 0:00:00


In [10]:
from selenium import webdriver
from webdriver_manager.chrome import ChromeDriverManager
from collections import defaultdict
from dataclasses import dataclass
from bs4 import BeautifulSoup
import pandas as pd
import requests
import json
from bottle import *
from pivottablejs import pivot_ui
from ast import literal_eval
import pandas as pd
import json
from bottle import *
import re
import requests
from urllib.request import Request,urlopen
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.wait import WebDriverWait
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options


@dataclass
class Event:
    imgSrc : list
    pokemonName : list
    combatPower : list
    shiny : list
    pokeId: list

    def to_dict(self):
        metadata = {
                "imgSrc": self.imgSrc,
                "pokemonName": self.pokemonName,
                "pokeId" : self.pokeId,
                "shiny": self.shiny,
                "combatPower": self.combatPower,
        }
        return metadata

    def __str__(self):
        return str(self.to_dict())

def main():
    events = defaultdict()
    service = Service(executable_path=r'/usr/bin/chromedriver')
    options = webdriver.ChromeOptions()
    options.add_argument("--headless")
    options.add_argument("--no-sandbox")

    options.headless = True

    driver = webdriver.Chrome(service=service, options=options)

    distance =[]
    imgSrc=[]
    pokemonName=[]
    combatPower=[]
    data=[]
    distanceEgg=[]
    summaries=[]
    imgSrcEgg=[]
    pokemonNameEgg=[]
    combatPowerEgg=[]
    shinyEgg=[]
    shiny=[]
    pokeId=[]
    pokeIdEgg=[]

    url = "https://leekduck.com/eggs/"

    driver.get(url)

    soup = BeautifulSoup(driver.page_source, "lxml")

    for item in soup.find_all('h2'):
        distance.append(item.string)
    #distance2=distance.pop()
    newdis=distance[:7]
    for i in range(len(newdis)-1):
        soup1=soup.find_all("ul",class_='egg-list-flex')[i]
        soup2=soup1.find_all("li",class_='egg-list-item')
        for li in soup2:
            img=li.find('img')
            imgShiny=li.find("img",class_='shiny-icon')
            if imgShiny!=None:
                shiny.append(True)
            else:
                shiny.append(False)
            src=img.get('src')
            if src:
                src = requests.compat.urljoin(url, src)
                imgSrc.append(src)
            span=li.find("span",class_='hatch-pkmn')
            pokemonName.append(span.string)
            spanCp=li.find("div",class_='font-size-smaller color-555555')
            spanCpStr=str(spanCp)
            spanCpList=spanCpStr.split('n>')[1]
            spanCpCleaned=str(spanCpList).replace('</div>','')
            combatPower.append(spanCpCleaned.replace('\n','').strip())
        for h in pokemonName:
            pokeName=h.lower()
            if pokeName == 'mime jr.':
                pokeName = 'mr-mime-galar'
            elif 'far' in pokeName:
                pokeName = 'farfetchd-galar'
            elif 'alolan' in pokeName:
                pokeNamelist= pokeName.split(' ')
                pokeNamelist=pokeNamelist[1]
                pokeName=pokeNamelist+'-alola'
            elif 'galarian' in pokeName:
                pokeNamelist= pokeName.split(' ')
                pokeNamelist=pokeNamelist[1]
                pokeName=pokeNamelist+'-galar'
            elif 'hisuian' in pokeName:
                pokeNamelist= pokeName.split(' ')
                pokeNamelist=pokeNamelist[1]
                pokeName=pokeNamelist+'-hisui'
            url='https://pokeapi.co/api/v2/pokemon/'
            newurl=url+pokeName
            req=Request(
                url=newurl,
                headers={'User-Agent': 'Mozilla/5.0'}
            )
            responseJSON2=urlopen(req)
            dataJSONType=json.loads(responseJSON2.read())
            pokeId.append(dataJSONType['id'])
        events[distance[i]]=Event(imgSrc,pokemonName,combatPower,shiny,pokeId)
        imgSrc=[]
        pokemonName=[]
        combatPower=[]
        shiny=[]
        pokeId=[]

    df=pd.DataFrame(events,index=[0]).T
    df.columns=['Summary']

    df.to_csv('./egg_data.csv')

    df=pd.read_csv('./egg_data.csv')
    eggData=pd.DataFrame()

    for i in range(len(df)):
        distanceEgg.append(df.iloc[i,0])
        summaries.append(df.iloc[i,1])

    eggData['Distance'] = distanceEgg

    for summary in summaries:
        summ=summary.split('pokemonName')
        summ=summ[0].split(':',1)
        strToBeCleaned=summ[1]
        strToReplace=','
        replacementStr=''
        pos=strToBeCleaned.rfind(strToReplace)
        if pos >-1:
            summCleanedSplit=strToBeCleaned[:pos]+replacementStr + strToBeCleaned[pos + len(strToReplace): ]
        else:
            summCleanedSplit=strToBeCleaned
        strToBeCleaned2=summCleanedSplit
        strToReplace2='\''
        replacementStr2=''
        pos2=strToBeCleaned2.rfind(strToReplace2)
        if pos2 >-1:
            summCleanedSplit=strToBeCleaned2[:pos2]+replacementStr2 + strToBeCleaned2[pos2 + len(strToReplace2): ]
        else:
            summCleanedSplit=strToBeCleaned2

        imgSrcEgg.append(summCleanedSplit)

    for  summary in summaries:
        summ=summary.split('pokemonName')
        summ=summ[1].split('combatPower')
        summ=summ[0].split(':')
        summ=summ[1].rsplit(',',1)
        strToBeCleaned=summ[0]
        """strToReplace=','
        replacementStr=''
        pos=strToBeCleaned.rfind(strToReplace)
        if pos >-1:
            summCleanedSplit=strToBeCleaned[:pos]+replacementStr + strToBeCleaned[pos + len(strToReplace): ]
        else:
            summCleanedSplit=strToBeCleaned
        #strToBeCleaned2=summCleanedSplit
        strToBeCleaned2=strToBeCleaned
        strToReplace2='\''
        replacementStr2=''
        pos2=strToBeCleaned2.rfind(strToReplace2)
        if pos2 >-1:
            summCleanedSplit=strToBeCleaned2[:pos2]+replacementStr2 + strToBeCleaned2[pos2 + len(strToReplace2): ]
        else:
            summCleanedSplit=strToBeCleaned2"""
        pokemonNameEgg.append(strToBeCleaned)

    for summary in summaries:
        summ=summary.split('combatPower')
        summ=summ[1].split(':',1)
        summ=summ[1].replace('}','')
        combatPowerEgg.append(summ)

    for summary in summaries:
        summ=summary.split('shiny')
        summ=summ[1].split('combatPower')
        summ=summ[0].split(':',1)
        summ=summ[1].rsplit(',',1)
        shinyEgg.append(summ[0])

    for summary in summaries:
        summ=summ=summary.split('pokeId')
        summ=summ[1].split('shiny')
        summ=summ[0].split(':',1)
        summ=summ[1].rsplit(',',1)
        pokeIdEgg.append(summ[0])


    eggData['imgSrc'] = imgSrcEgg
    eggData['pokemonName'] = pokemonNameEgg
    eggData['pokeId'] = pokeIdEgg
    eggData['shiny'] = shinyEgg
    eggData['combatPower'] = combatPowerEgg

    eggData['json'] = eggData.to_json(orient='records', lines=True).splitlines()

    pivot_ui(eggData)

    for i in eggData['json']:
        data.append(json.loads(i))

    for i in data:
        i['imgSrc'] = literal_eval(i['imgSrc'])
        i['pokemonName'] = literal_eval(i['pokemonName'])
        i['combatPower'] = literal_eval(i['combatPower'])
        i['shiny'] = literal_eval(i['shiny'])
        i['pokeId'] = literal_eval(i['pokeId'])

    with open('./egg_data.json',"w") as output_file:
        #output_file.write(pokemon_json.replace('\\','')+'\n')
        #output_file.write(pokemon_json_cleaned)
        #output_file.write(json.dumps({"data": pokemonJsonCleaned}, indent=4 ))
        output_file.write(json.dumps({"data": data}, indent=4 ))
        driver.quit()

if __name__ == "__main__":
    main()
    url = "https://getpantry.cloud/apiv1/pantry/b45d3e57-17a6-498d-8aec-b8173408efb4/basket/eggData"

    f = open('/content/egg_data.json')

    payload = json.load(f)
    payload=json.dumps(payload)
    headers = {
      'Content-Type': 'application/json'
    }

    response = requests.request("POST", url, headers=headers, data=payload)

    print(response.text)

<ipython-input-10-103279974995>:54: DeprecationWarning: headless property is deprecated, instead use add_argument('--headless') or add_argument('--headless=new')
  options.headless = True


Your Pantry was updated with basket: eggData!
